# 🧹 Notebook 02 — Data Cleaning & ETL Pipeline
### HR Analytics: Job Change of Data Scientists
**Objective:** Full production-grade ETL — imputation, encoding, feature engineering, quality validation.
**Output:** `data/processed/hr_cleaned.csv` | `hr_engineered.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'primary': '#1B3A6B', 'accent': '#E84855', 'safe': '#2ECC71'}

RAW_PATH    = '../data/raw/aug_train.csv'
CLEAN_PATH  = '../data/processed/hr_cleaned.csv'
ENG_PATH    = '../data/processed/hr_engineered.csv'

df_raw = pd.read_csv(RAW_PATH)
print(f'Raw shape: {df_raw.shape}')
df = df_raw.copy()

## 1. Structural Fixes

In [ ]:
# Fix whitespace in column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Strip string values
for col in df.select_dtypes('object').columns:
    df[col] = df[col].str.strip()

# Drop duplicate enrollee_ids
n_before = len(df)
df = df.drop_duplicates(subset='enrollee_id')
print(f'Duplicates removed: {n_before - len(df)}')
print(f'Shape after dedup: {df.shape}')

## 2. Missing Value Treatment

In [ ]:
# ── HIGH missingness: company fields → 'Unknown' (likely not employed) ──────
df['company_size'] = df['company_size'].fillna('Unknown')
df['company_type'] = df['company_type'].fillna('Unknown')

# ── MEDIUM missingness: gender → 'Unknown' (sensitive field) ─────────────────
df['gender'] = df['gender'].fillna('Unknown')

# ── MEDIUM missingness: major_discipline ─────────────────────────────────────
# Primary/High school students likely have 'No Major'
no_major_mask = df['education_level'].isin(['Primary School', 'High School'])
df.loc[no_major_mask & df['major_discipline'].isnull(), 'major_discipline'] = 'No Major'
df['major_discipline'] = df['major_discipline'].fillna(df['major_discipline'].mode()[0])

# ── LOW missingness: mode imputation ─────────────────────────────────────────
for col in ['enrolled_university', 'education_level', 'experience', 'last_new_job']:
    df[col] = df[col].fillna(df[col].mode()[0])

# ── Verify: no missing values remain ─────────────────────────────────────────
remaining_null = df.isnull().sum()
print('Remaining nulls after imputation:')
print(remaining_null[remaining_null > 0] if remaining_null.sum() > 0 else '  ✅ Zero missing values!')

## 3. Ordinal Encoding

In [ ]:
# ── Experience: string ordinal → numeric ─────────────────────────────────────
exp_map = {f'{i}': i for i in range(1, 21)}
exp_map['<1'] = 0
exp_map['>20'] = 21
df['experience_num'] = df['experience'].map(exp_map).astype(float)

# ── Education level: ordinal ──────────────────────────────────────────────────
edu_map = {
    'Primary School': 1,
    'High School':    2,
    'Graduate':       3,
    'Masters':        4,
    'Phd':            5
}
df['education_ordinal'] = df['education_level'].map(edu_map)

# ── Last new job: ordinal ─────────────────────────────────────────────────────
lnj_map = {'never': 0, '1': 1, '2': 2, '3': 3, '4': 4, '>4': 5}
df['last_new_job_num'] = df['last_new_job'].map(lnj_map).astype(float)

# ── Company size: ordinal ─────────────────────────────────────────────────────
cs_map = {
    'Unknown':   0,
    '<10':       1,
    '10/49':     2,
    '50-99':     3,
    '100-500':   4,
    '500-999':   5,
    '1000-4999': 6,
    '5000-9999': 7,
    '10000+':    8
}
df['company_size_num'] = df['company_size'].map(cs_map)

# ── Relevant experience: binary ───────────────────────────────────────────────
df['has_relevent_exp'] = (df['relevent_experience'] == 'Has relevent experience').astype(int)

print('Ordinal encoding complete.')
df[['experience', 'experience_num', 'education_level', 'education_ordinal',
    'last_new_job', 'last_new_job_num', 'company_size', 'company_size_num']].head(10)

## 4. Outlier Treatment

In [ ]:
# ── training_hours: IQR-based capping ────────────────────────────────────────
Q1  = df['training_hours'].quantile(0.25)
Q3  = df['training_hours'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 3 * IQR   # Use 3×IQR (lenient) — real-world training can be long

n_outliers = (df['training_hours'] > upper_fence).sum()
df['training_hours_capped'] = df['training_hours'].clip(upper=upper_fence)

print(f'training_hours — Q1:{Q1}, Q3:{Q3}, IQR:{IQR}')
print(f'Upper fence (3×IQR): {upper_fence:.1f}')
print(f'Outliers capped: {n_outliers} rows ({n_outliers/len(df)*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['training_hours'],        bins=40, color=COLORS['accent'], alpha=0.8, edgecolor='white')
axes[0].set_title('Before Capping')
axes[1].hist(df['training_hours_capped'], bins=40, color=COLORS['safe'],   alpha=0.8, edgecolor='white')
axes[1].set_title('After Capping (3×IQR)')
for ax in axes: ax.set_xlabel('Training Hours')
plt.suptitle('Outlier Treatment — training_hours', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/02_outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Engineering

In [ ]:
# ── Experience Band ───────────────────────────────────────────────────────────
def experience_band(x):
    if pd.isna(x): return 'Unknown'
    if x == 0:     return 'Fresher (<1yr)'
    if x <= 3:     return 'Junior (1-3yr)'
    if x <= 7:     return 'Mid (4-7yr)'
    if x <= 15:    return 'Senior (8-15yr)'
    return 'Veteran (>15yr)'
df['experience_band'] = df['experience_num'].apply(experience_band)

# ── City Tier based on CDI ────────────────────────────────────────────────────
def city_tier(cdi):
    if cdi >= 0.80: return 'Tier 1 (Developed)'
    if cdi >= 0.60: return 'Tier 2 (Developing)'
    return 'Tier 3 (Underdeveloped)'
df['city_tier'] = df['city_development_index'].apply(city_tier)

# ── STEM Flag ─────────────────────────────────────────────────────────────────
df['is_stem'] = (df['major_discipline'] == 'STEM').astype(int)

# ── University Enrollment Binary ──────────────────────────────────────────────
df['is_enrolled'] = (df['enrolled_university'] != 'no_enrollment').astype(int)

# ── Retention Risk Score (composite) ─────────────────────────────────────────
# Higher score = higher attrition risk
# Logic: low CDI + low company_size + low experience + no relevant exp = high risk
df['retention_risk_score'] = (
    (1 - df['city_development_index']) * 3 +          # CDI inverse
    (8 - df['company_size_num'].clip(0, 8)) * 0.5 +   # smaller company = higher risk
    (21 - df['experience_num'].clip(0, 21)) * 0.1 +   # less exp = higher risk
    (1 - df['has_relevent_exp']) * 1.0 +               # no relevant exp
    (1 - df['is_stem']) * 0.5                          # non-STEM slightly higher risk
).round(4)

# Normalize to 0–10
rrs_min = df['retention_risk_score'].min()
rrs_max = df['retention_risk_score'].max()
df['retention_risk_score'] = ((df['retention_risk_score'] - rrs_min) / (rrs_max - rrs_min) * 10).round(2)

# ── Risk Tier ─────────────────────────────────────────────────────────────────
df['risk_tier'] = pd.cut(
    df['retention_risk_score'],
    bins=[0, 3.5, 6.5, 10],
    labels=['Low Risk', 'Medium Risk', 'High Risk'],
    include_lowest=True
)

print('Feature engineering complete. New columns:')
print(['experience_band', 'city_tier', 'is_stem', 'is_enrolled', 'retention_risk_score', 'risk_tier'])
df[['enrollee_id', 'city_development_index', 'city_tier', 'experience_num',
    'experience_band', 'retention_risk_score', 'risk_tier', 'target']].head(10)

## 6. One-Hot Encoding for Modelling

In [ ]:
# Keep a clean analytical version before OHE
df_clean = df.copy()
df_clean.to_csv(CLEAN_PATH, index=False)
print(f'[SAVED] hr_cleaned.csv — shape: {df_clean.shape}')

# Columns to one-hot encode for modelling
ohe_cols = ['gender', 'enrolled_university', 'major_discipline', 'company_type', 'city_tier']

df_eng = df_clean.copy()
df_eng = pd.get_dummies(df_eng, columns=ohe_cols, drop_first=True, dtype=int)

# Drop high-cardinality / redundant cols before modelling
drop_cols = ['enrollee_id', 'city', 'relevent_experience', 'experience', 'education_level',
             'last_new_job', 'company_size', 'training_hours', 'experience_band', 'risk_tier']
df_eng = df_eng.drop(columns=[c for c in drop_cols if c in df_eng.columns])

df_eng.to_csv(ENG_PATH, index=False)
print(f'[SAVED] hr_engineered.csv — shape: {df_eng.shape}')
df_eng.head()

## 7. Data Quality Validation Report

In [ ]:
print('='*60)
print('  ETL VALIDATION REPORT')
print('='*60)
print(f'  Raw rows          : {len(df_raw):,}')
print(f'  Cleaned rows      : {len(df_clean):,}')
print(f'  Rows lost         : {len(df_raw)-len(df_clean)}')
print(f'  Remaining nulls   : {df_clean.isnull().sum().sum()}')
print(f'  New features      : 6 engineered columns')
print(f'  Target preserved  : {df_clean["target"].value_counts().to_dict()}')
print('='*60)
print('\n  Feature Summary:')
for col in ['experience_band', 'city_tier', 'risk_tier']:
    print(f'\n  {col}:')
    print(df_clean[col].value_counts().to_string(header=False))
print('\n✅ ETL complete. Proceeding to 03_eda.ipynb')

---
## ✅ Next: `03_eda.ipynb` — Exploratory Data Analysis